# LlamaIndex와 AgentCore Memory - 학술 연구 Assistant(단기 메모리)

## 소개

이 Notebook에서는 Amazon Bedrock AgentCore Memory 기능을 LlamaIndex와 통합하여 학술 연구 Assistant를 만드는 방법을 살펴봅니다. 하나의 연구 세션 안에서 **단기 메모리**를 유지하여 대화 전반에 걸쳐 논문, 연구 결과, 연구 컨텍스트를 기억하도록 하는 데 중점을 둡니다.

## 튜토리얼 세부 정보

| 정보         | 세부 정보                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 단기 대화 메모리                                                |
| Agent 사용 사례       | 학술 연구 Assistant                                                      |
| Agentic Framework   | LlamaIndex                                                                       |
| LLM 모델           | Anthropic Claude 3.7 Sonnet                                                  |
| 튜토리얼 구성 요소 | AgentCore Short-term Memory, LlamaIndex Agent, 연구 도구                   |
| 예제 난이도  | 초급                                                                         |

다음 내용을 학습합니다.
- 연구 데이터를 유지하기 위한 AgentCore Memory 생성
- LlamaIndex 기본 메모리 통합 사용
- 논문 분석을 위한 연구 전용 도구 구축
- 단일 세션 내에서 연구 컨텍스트 유지
- 메모리 경계 및 세션 격리 테스트

## 시나리오 배경

이 예제에서는 연구자가 단일 연구 세션 안에서 논문, 연구 결과, 연구 주제를 추적하도록 돕는 "학술 연구 Assistant"를 만듭니다. Assistant는 AgentCore Memory를 사용하여 대화 전반에 걸쳐 검토한 논문, 발견한 주요 결과, 연구 진행 상황에 관한 컨텍스트를 유지합니다.

## 아키텍처 개요

![LlamaIndex AgentCore Short-Term Memory Architecture](LlamaIndex-AgentCore-STM-Arch.png)

## 사전 요구 사항

- Python 3.10 이상
- 적절한 권한이 있는 AWS 계정
- AgentCore Memory 권한이 있는 AWS IAM 역할:
  - `bedrock-agentcore:CreateMemory`
  - `bedrock-agentcore:CreateEvent`
  - `bedrock-agentcore:ListEvents`
  - `bedrock-agentcore:RetrieveMemories`
- Amazon Bedrock 모델에 대한 액세스

## 1단계: 종속성 설치 및 설정

In [ ]:
# 필요한 라이브러리 설치
%pip install llama-index-memory-bedrock-agentcore llama-index-llms-bedrock-converse boto3

In [ ]:
# 필요한 구성 요소 가져오기
from bedrock_agentcore.memory import MemoryClient
from llama_index.memory.bedrock_agentcore import AgentCoreMemory, AgentCoreMemoryContext
from llama_index.llms.bedrock_converse import BedrockConverse
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.tools import FunctionTool
from datetime import datetime
import os

## 2단계: AgentCore Memory 구성

연구 Assistant에서 사용할 AgentCore Memory 리소스를 생성하거나 가져옵니다.

In [ ]:
# AgentCore Memory 리소스 생성
region = os.getenv("AWS_REGION", "us-east-1")
client = MemoryClient(region_name=region)

try:
    response = client.create_memory_and_wait(
        name=f"AcademicResearchShortTerm_{int(datetime.now().timestamp())}",
        description="Academic research assistant short-term memory for single session context",
        strategies=[],
        event_expiry_days=7,
        max_wait=300,
        poll_interval=10,
    )
    memory_id = response["id"]
    print(f"✅ Created AgentCore Memory: {memory_id}")
except Exception as e:
    print(f"❌ Error creating memory: {e}")
    memory_id = "your-memory-id-here"  # 기존 Memory ID로 교체

## 3단계: 연구 도구 구현

학술 연구 작업을 위한 전문 도구를 정의합니다.

In [ ]:
def save_paper_summary(title: str, authors: str, key_findings: str) -> str:
    """Save a research paper summary with title, authors, and key findings"""
    print(f"📄 Saved paper: {title} by {authors}")
    return f"Successfully saved paper summary for '{title}'"


def track_research_topic(topic: str, status: str) -> str:
    """Track research topic progress with current status"""
    print(f"🔬 Tracking research topic: {topic} (Status: {status})")
    return f"Now tracking research topic: {topic} with status {status}"


def save_research_finding(finding: str, confidence: str) -> str:
    """Save a research finding with confidence level"""
    print(f"💡 Research finding saved with {confidence} confidence")
    return f"Saved research finding with {confidence} confidence level"


# Agent용 도구 객체 생성
research_tools = [
    FunctionTool.from_defaults(fn=save_paper_summary),
    FunctionTool.from_defaults(fn=track_research_topic),
    FunctionTool.from_defaults(fn=save_research_finding),
]

## 4단계: LlamaIndex Agent 구현

단기 메모리 컨텍스트를 사용하는 연구 Assistant Agent를 생성합니다.

In [ ]:
# 단기 메모리 구성(단일 세션)
MODEL_ID = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"

# 단일 세션용 메모리 컨텍스트 생성
context = AgentCoreMemoryContext(
    actor_id="academic-researcher",
    memory_id=memory_id,
    session_id="research-session-today",  # 전체 과정에서 동일한 세션 사용
    namespace="/academic-research/",
)

# AgentCore Memory 및 LLM 초기화
agentcore_memory = AgentCoreMemory(context=context)
llm = BedrockConverse(model=MODEL_ID)

# 연구 Assistant Agent 생성
research_agent = FunctionAgent(tools=research_tools, llm=llm, verbose=True)

print("✅ Academic Research Assistant with short-term memory is ready!")

## 5단계: 단기 메모리 기능 테스트

종합 연구 세션을 통해 연구 Assistant의 단기 메모리를 테스트해 보겠습니다.

### 테스트 1: 세션 초기화

In [ ]:
# 세부 컨텍스트로 연구 세션 초기화
response = await research_agent.run(
    "I'm Dr. Sarah Smith from MIT's Computer Science Department, starting research on 'Machine Learning in Healthcare Applications'. "
    "Track this topic with status 'Literature Review'.",
    memory=agentcore_memory,
)

print("🎯 Session Initialization:")
print(response)

### 테스트 2: 연구 논문 추가

In [ ]:
# 세부 지표가 포함된 첫 번째 논문 추가
response = await research_agent.run(
    "Save paper: 'Deep Learning for Medical Image Analysis' by Zhang et al. "
    "Key findings: CNNs achieve 95.2% accuracy in chest X-ray diagnosis, 12% improvement over radiologists, "
    "trained on 100,000 images with 0.03 false positive rate.",
    memory=agentcore_memory,
)

print("📄 Paper 1 Added:")
print(response)

In [ ]:
# 상반된 결과가 포함된 두 번째 논문 추가
response = await research_agent.run(
    "Save paper: 'Transformers in Medical NLP' by Johnson et al. "
    "Key findings: BERT models achieve 89.1% F1-score in clinical note classification, "
    "struggle with rare diseases (<70% accuracy), excel at symptom extraction (94% precision).",
    memory=agentcore_memory,
)

print("📄 Paper 2 Added:")
print(response)

### 테스트 3: 신원 및 컨텍스트 회상

In [ ]:
# 신원 및 연구 컨텍스트 회상 테스트
response = await research_agent.run("What's my name, institution, and current research focus?", memory=agentcore_memory)

print("🧠 Identity Recall Test:")
print(response)
print("\n✅ Expected: Dr. Sarah Smith, MIT, Machine Learning in Healthcare")

### 테스트 4: 세부 지표 회상

In [ ]:
# 특정 지표 회상 테스트
response = await research_agent.run(
    "What were the exact accuracy percentages mentioned in the papers I reviewed? "
    "Which authors wrote about CNNs vs Transformers?",
    memory=agentcore_memory,
)

print("📊 Detailed Metrics Recall:")
print(response)
print("\n✅ Expected: Zhang et al - CNNs 95.2%, Johnson et al - BERT 89.1%")

### 테스트 5: 컨텍스트 기반 추론

In [ ]:
# 컨텍스트 이해 및 추론 테스트
response = await research_agent.run(
    "Based on the papers I've reviewed, which approach would be better for analyzing "
    "chest X-rays vs clinical notes? Explain your reasoning.",
    memory=agentcore_memory,
)

print("🤔 Contextual Reasoning Test:")
print(response)
print("\n✅ Expected: CNNs for X-rays (Zhang paper), Transformers for clinical notes (Johnson paper)")

### 테스트 6: 연구 결과 종합

In [ ]:
# 종합한 연구 결과 추가
response = await research_agent.run(
    "Based on Zhang's CNN results (95.2% accuracy) and Johnson's Transformer results (89.1% F1-score), "
    "I conclude that deep learning models consistently achieve >85% accuracy in healthcare tasks. "
    "This finding has high confidence. Save it.",
    memory=agentcore_memory,
)

print("🔬 Research Finding Synthesis:")
print(response)

### 테스트 7: 상호 참조 기능

In [ ]:
# 연구 결과와 논문 간 상호 참조 테스트
response = await research_agent.run(
    "How does my research finding about >85% accuracy relate to the specific results "
    "from Zhang and Johnson? What evidence supports this conclusion?",
    memory=agentcore_memory,
)

print("🔗 Cross-Reference Test:")
print(response)
print("\n✅ Expected: Reference to Zhang 95.2% and Johnson 89.1% as supporting evidence")

### 테스트 8: 실제 적용 시나리오

In [ ]:
# 축적된 지식의 실제 적용 테스트
response = await research_agent.run(
    "I'm writing a grant proposal for healthcare AI research. What evidence can I cite "
    "about deep learning effectiveness? Include specific numbers and authors.",
    memory=agentcore_memory,
)

print("📝 Grant Proposal Support:")
print(response)
print("\n✅ Expected: Comprehensive summary with Zhang 95.2%, Johnson 89.1%, synthesis finding")

## 6단계: 세션 경계 테스트

별도의 세션을 생성하여 단기 메모리의 경계를 테스트해 보겠습니다.

In [ ]:
# 별도의 세션 컨텍스트 생성
new_session_context = AgentCoreMemoryContext(
    actor_id="academic-researcher",
    memory_id=memory_id,
    session_id="different-research-session",  # 서로 다른 세션 ID
    namespace="/academic-research/",
)

new_session_memory = AgentCoreMemory(context=new_session_context)

# 메모리 격리 테스트
response = await research_agent.run(
    "What research have I been working on? What specific accuracy numbers did I find?",
    memory=new_session_memory,
)

print("🚧 Session Boundary Test (Different Session):")
print(response)
print("\n✅ Expected: Limited or no recall from previous session (short-term memory boundary)")

In [ ]:
# 지속성을 검증하기 위해 원래 세션으로 복귀
response = await research_agent.run(
    "Now back in my original session - what were the accuracy numbers from Zhang and Johnson again?",
    memory=agentcore_memory,  # 원래 세션 메모리
)

print("🔄 Original Session Return:")
print(response)
print("\n✅ Expected: Full recall of Zhang 95.2%, Johnson 89.1%")

## 🧪 자동 테스트 검증
다음 셀을 실행하여 메모리 통합이 올바르게 작동하는지 검증합니다.

In [ ]:
# 검증 함수를 인라인으로 정의
class TestValidator:
    def __init__(self):
        self.results = {}

    def validate_memory_recall(self, response):
        """에이전트가 세션 앞부분의 정보를 기억하는지 확인합니다."""
        # 실질적인 응답인지 확인("I don't know"만 있는 응답 제외)
        has_content = len(response) > 50
        # 메모리 관련 표현 확인
        has_memory_indicators = any(
            word in response.lower()
            for word in [
                "earlier",
                "mentioned",
                "discussed",
                "previously",
                "you",
                "we",
                "our",
            ]
        )
        return "✅ PASS" if (has_content and has_memory_indicators) else "❌ FAIL"

    def validate_session_memory(self, response):
        """에이전트가 세션 내 컨텍스트를 유지하는지 확인합니다."""
        has_memory_content = len(response) > 100 and any(
            word in response.lower()
            for word in [
                "previous",
                "earlier",
                "mentioned",
                "discussed",
                "before",
                "already",
            ]
        )
        return "✅ PASS" if has_memory_content else "❌ FAIL"

    def validate_cross_reference(self, response):
        """에이전트가 현재 질의를 이전 컨텍스트와 연결할 수 있는지 확인합니다."""
        # 연결 표현 확인
        connecting_words = [
            "relate",
            "connection",
            "previous",
            "earlier",
            "discussed",
            "mentioned",
            "context",
            "based on",
            "as we",
            "as i",
        ]
        has_connection = any(word in response.lower() for word in connecting_words)
        has_substance = len(response) > 80
        return "✅ PASS" if (has_connection and has_substance) else "❌ FAIL"

    def run_validation_summary(self, test_results):
        print("🧪 COMPREHENSIVE TEST VALIDATION SUMMARY")
        print("=" * 60)

        total_tests = len(test_results)
        passed_tests = sum(1 for result in test_results.values() if "PASS" in result)
        pass_rate = (passed_tests / total_tests * 100) if total_tests > 0 else 0

        for test_name, result in test_results.items():
            print(f"{test_name}: {result}")

        print("=" * 60)
        print(f"📊 Overall Pass Rate: {passed_tests}/{total_tests} ({pass_rate:.1f}%)")

        if pass_rate >= 80:
            print("✅ EXCELLENT: Memory integration working correctly!")
        elif pass_rate >= 60:
            print("⚠️  GOOD: Most memory features working, some issues to investigate")
        else:
            print("❌ NEEDS ATTENTION: Memory integration has significant issues")

        return pass_rate


validator = TestValidator()
print("✅ Validation functions loaded!")

In [ ]:
# 모든 검증 테스트 실행
test_results = {}

# 테스트 1: 메모리 회상 - Agent가 논의한 내용을 기억하는가?
response1 = await research_agent.run("What have we discussed so far in this session?", memory=agentcore_memory)
test_results["Memory Recall"] = validator.validate_memory_recall(str(response1))
print(f"Response 1 length: {len(str(response1))} chars")

# 테스트 2: 세션 메모리 - Agent가 컨텍스트를 유지하는가?
response2 = await research_agent.run("What did we talk about earlier?", memory=agentcore_memory)
test_results["Session Memory"] = validator.validate_session_memory(str(response2))
print(f"Response 2 length: {len(str(response2))} chars")

# 테스트 3: 상호 참조 기능 - Agent가 이전 컨텍스트와 연결할 수 있는가?
response3 = await research_agent.run("How does this relate to what we discussed before?", memory=agentcore_memory)
test_results["Cross Reference"] = validator.validate_cross_reference(str(response3))
print(f"Response 3 length: {len(str(response3))} chars")

# 결과 표시
validator.run_validation_summary(test_results)

## 요약

이 Notebook에서는 다음 내용을 구현했습니다.

✅ **단기 메모리 통합**: LlamaIndex에서 AgentCore Memory를 사용하여 세션 범위의 데이터 유지

✅ **연구 전용 도구**: 논문 요약, 주제 추적, 연구 결과 저장

✅ **컨텍스트 기반 대화**: Assistant가 세션 내의 세부 정보를 기억

✅ **상호 참조 기능**: 여러 논문과 상호작용의 연구 결과를 연결

✅ **세션 경계**: 서로 다른 대화 세션 간의 메모리 격리

✅ **실제 적용**: 연구비 제안서 지원 및 연구 결과 종합

학술 연구 Assistant는 단기 메모리를 통해 하나의 연구 세션 안에서 자연스럽고 컨텍스트에 맞는 대화를 제공하는 동시에 서로 다른 대화 thread 사이의 경계를 명확히 유지하는 방법을 보여 줍니다.

## 리소스 정리

이 Notebook에서 사용한 리소스를 정리하기 위해 Memory를 삭제합니다.

In [ ]:
# AgentCore Memory 리소스 정리
try:
    client.delete_memory(memory_id)
    print(f"✅ Successfully deleted memory: {memory_id}")
except Exception as e:
    print(f"❌ Error deleting memory: {e}")